# Stage-2 Classification Results — Real-Time Evaluation\n\nThis notebook loads the actual EXP-11 RunB2 ResNet-50 checkpoint and runs real inference on both the **Validation** and **Test** classifier crop sets.\nAll metrics are computed in real-time — nothing is hardcoded.\n\n**Model:** EXP-11 RunB2 ResNet-50 (224x224, Inverse-Freq Class Weights)\n**Checkpoint:** `runs/classify/EXP-11-RunB2/best.pt`

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\n!pip install torchvision scikit-learn -q

In [ ]:
import os\nimport torch\nimport torch.nn as nn\nfrom torchvision import transforms, datasets, models\nfrom torch.utils.data import DataLoader\nfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport numpy as np\n\nsns.set_theme(style='whitegrid')\nplt.rcParams['figure.dpi'] = 120\n\nPROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'\nos.chdir(PROJECT_ROOT)\nprint(f'Working directory: {os.getcwd()}')\nprint(f'CUDA: {torch.cuda.is_available()}')

## 1. Load EXP-11 RunB2 ResNet-50 Checkpoint

In [ ]:
# Define the model architecture (must match training)\nCLASS_NAMES = ['Inclusion-Particle', 'Porosity', 'Tear-Delamination']\nNUM_CLASSES = 3\n\n# Build ResNet-50 with custom head\nresnet_model = models.resnet50(weights=None)\nresnet_model.fc = nn.Linear(resnet_model.fc.in_features, NUM_CLASSES)\n\n# Load the trained checkpoint\nRESNET_CKPT = 'runs/classify/EXP-11-RunB2/best.pt'\nassert os.path.exists(RESNET_CKPT), f'Checkpoint NOT found at {RESNET_CKPT}!'\n\nstate_dict = torch.load(RESNET_CKPT, map_location='cpu')\n# Handle different checkpoint formats\nif 'model_state_dict' in state_dict:\n    resnet_model.load_state_dict(state_dict['model_state_dict'])\nelif 'state_dict' in state_dict:\n    resnet_model.load_state_dict(state_dict['state_dict'])\nelse:\n    resnet_model.load_state_dict(state_dict)\n\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nresnet_model = resnet_model.to(device)\nresnet_model.eval()\nprint(f'ResNet-50 loaded successfully on {device}.')\nprint(f'Classes: {CLASS_NAMES}')

## 2. Prepare Validation & Test DataLoaders

In [ ]:
# Standard ImageNet normalization used during training\neval_transform = transforms.Compose([\n    transforms.Resize((224, 224)),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])\n])\n\nCROPS_DIR = 'datasets/classifier_crops'\n\nval_dataset = datasets.ImageFolder(os.path.join(CROPS_DIR, 'valid'), transform=eval_transform)\ntest_dataset = datasets.ImageFolder(os.path.join(CROPS_DIR, 'test'), transform=eval_transform)\n\nval_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)\ntest_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)\n\nprint(f'Validation samples: {len(val_dataset)}')\nprint(f'Test samples: {len(test_dataset)}')\nprint(f'Classes detected: {val_dataset.classes}')

## 3. Run Inference & Compute Metrics

In [ ]:
def evaluate_classifier(model, dataloader, class_names, split_name):\n    all_preds = []\n    all_labels = []\n    \n    with torch.no_grad():\n        for images, labels in dataloader:\n            images = images.to(device)\n            outputs = model(images)\n            _, predicted = torch.max(outputs, 1)\n            all_preds.extend(predicted.cpu().numpy())\n            all_labels.extend(labels.cpu().numpy())\n    \n    acc = accuracy_score(all_labels, all_preds) * 100\n    macro_f1 = f1_score(all_labels, all_preds, average='macro') * 100\n    weighted_f1 = f1_score(all_labels, all_preds, average='weighted') * 100\n    \n    print(f'\\n{"="*60}')\n    print(f'{split_name} SET RESULTS')\n    print(f'{"="*60}')\n    print(f'Accuracy:      {acc:.2f}%')\n    print(f'Macro F1:      {macro_f1:.2f}%')\n    print(f'Weighted F1:   {weighted_f1:.2f}%')\n    print(f'\\nPer-Class Report:')\n    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))\n    \n    # Confusion Matrix\n    cm = confusion_matrix(all_labels, all_preds)\n    fig, ax = plt.subplots(figsize=(8, 6))\n    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)\n    ax.set_title(f'{split_name} Set Confusion Matrix', fontsize=14)\n    ax.set_xlabel('Predicted', fontsize=12)\n    ax.set_ylabel('Actual', fontsize=12)\n    plt.tight_layout()\n    plt.savefig(f'classifier_{split_name.lower()}_confusion_matrix.png', dpi=150)\n    plt.show()\n    \n    return acc, macro_f1, weighted_f1, all_preds, all_labels

### 3A. Validation Set Evaluation

In [ ]:
val_acc, val_macro, val_weighted, val_preds, val_labels = evaluate_classifier(\n    resnet_model, val_loader, CLASS_NAMES, 'VALIDATION'\n)\nprint(f'\\nSource: Real-time inference using EXP-11-RunB2/best.pt checkpoint.')

### 3B. Test Set Evaluation

In [ ]:
test_acc, test_macro, test_weighted, test_preds, test_labels = evaluate_classifier(\n    resnet_model, test_loader, CLASS_NAMES, 'TEST'\n)\nprint(f'\\nSource: Real-time inference using EXP-11-RunB2/best.pt checkpoint.')

## 4. Validation vs Test Comparison

In [ ]:
comp_df = pd.DataFrame({\n    'Metric': ['Accuracy', 'Macro F1', 'Weighted F1'],\n    'Validation': [f'{val_acc:.2f}%', f'{val_macro:.2f}%', f'{val_weighted:.2f}%'],\n    'Test': [f'{test_acc:.2f}%', f'{test_macro:.2f}%', f'{test_weighted:.2f}%']\n})\ndisplay(comp_df)\n\n# Bar chart\nmetrics_names = ['Accuracy', 'Macro F1', 'Weighted F1']\nval_v = [val_acc, val_macro, val_weighted]\ntest_v = [test_acc, test_macro, test_weighted]\n\nx = np.arange(len(metrics_names))\nwidth = 0.35\nfig, ax = plt.subplots(figsize=(10, 6))\nb1 = ax.bar(x - width/2, val_v, width, label='Validation', color='#4CAF50')\nb2 = ax.bar(x + width/2, test_v, width, label='Test', color='#E91E63')\nax.set_ylabel('Percentage (%)')\nax.set_title('EXP-11 RunB2 ResNet-50: Validation vs Test Metrics', fontsize=14, pad=15)\nax.set_xticks(x)\nax.set_xticklabels(metrics_names, fontsize=11)\nax.legend()\nax.set_ylim(0, 100)\nfor bar in list(b1) + list(b2):\n    h = bar.get_height()\n    ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0,3), textcoords='offset points', ha='center', fontsize=9)\nplt.tight_layout()\nplt.savefig('classifier_val_vs_test.png', dpi=150)\nplt.show()

## 5. Summary\n\n**EXP-11 RunB2 ResNet-50** was selected as the final classifier because it achieved the highest validation Macro F1 score.\nAll metrics above are computed by running the actual checkpoint on real data — no hardcoded values.